# RFDet3D Training on Google Colab

Downloads ARKitScenes to Google Drive (persists across sessions),
then trains RFDet3D with wandb logging.

**First run:** Downloads data to Drive (~1-60GB depending on config).
**Subsequent runs:** Skips download, goes straight to training.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All data and checkpoints go here (persists across sessions)
DRIVE_ROOT = '/content/drive/MyDrive/RFDet3D'
!mkdir -p {DRIVE_ROOT}/data {DRIVE_ROOT}/ckpt {DRIVE_ROOT}/outputs

## 2. Install dependencies + clone repo

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version

In [ ]:
# Clone repo (or pull latest)
REPO_DIR = '/content/RFDet3D'

if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/Barath19/RFDet3D.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive

os.chdir(REPO_DIR)
!pwd

In [ ]:
# Install Python 3.11 + all deps
!uv python install 3.11
!uv sync

# Verify
!uv run python -c "import torch; print(f'torch={torch.__version__}, CUDA={torch.cuda.is_available()}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')"
!uv run python -c "import rfdetr; print('rfdetr OK')"

## 3. Download ARKitScenes to Google Drive

This only runs once — data persists on Drive.

| Setting | Scenes | Download | Disk on Drive |
|---------|--------|----------|---------------|
| `max_scenes=10` | 10 | ~1 GB | ~500 MB |
| `max_scenes=50` | 50 | ~6 GB | ~3 GB |
| `split=Validation` | 549 | ~60 GB | ~30 GB |
| `split=Training` | 4498 | ~560 GB | ~280 GB |

In [ ]:
#@title ARKitScenes Download Config { run: "auto" }
SPLIT = "Validation"  #@param ["Validation", "Training", "both"]
MAX_SCENES = 50  #@param {type: "integer"}
FRAME_STEP = 10  #@param {type: "integer"}
MAX_FRAMES_PER_SCENE = 50  #@param {type: "integer"}

DATA_DIR = f"{DRIVE_ROOT}/data/arkitscenes"
RAW_DIR = f"{DRIVE_ROOT}/data/arkitscenes_raw"

# Check if already downloaded
import glob
existing = glob.glob(f"{DATA_DIR}/images/*.jpg")
print(f"Existing images on Drive: {len(existing)}")

if len(existing) == 0:
    print(f"\nDownloading ARKitScenes ({SPLIT}, max {MAX_SCENES} scenes) to Google Drive...")
    print("This persists across Colab sessions.")
    print("Go get coffee — this takes a while on first run.\n")

    !uv run python scripts/prepare_arkitscenes.py \
        --split {SPLIT} \
        --max_scenes {MAX_SCENES} \
        --output_dir {DATA_DIR} \
        --download_dir {RAW_DIR} \
        --frame_step {FRAME_STEP} \
        --max_frames_per_scene {MAX_FRAMES_PER_SCENE}
else:
    print(f"Data already on Drive. Skipping download.")
    print(f"To re-download, delete {DATA_DIR} and re-run this cell.")

In [ ]:
# Verify data
import json

ann_files = glob.glob(f"{DATA_DIR}/*_annotations.json")
for f in ann_files:
    with open(f) as fp:
        data = json.load(fp)
    print(f"{os.path.basename(f)}: {len(data['images'])} images, {len(data['annotations'])} annotations, {len(data['categories'])} classes")

images = glob.glob(f"{DATA_DIR}/images/*")
depths = glob.glob(f"{DATA_DIR}/depth/*")
print(f"\nImage files: {len(images)}")
print(f"Depth files: {len(depths)}")

## 4. (Optional) Copy data to local SSD for faster training

Google Drive I/O is slow. Copy to Colab's local disk for 5-10x faster data loading.
This needs to be re-done each session but takes only a few minutes.

In [ ]:
USE_LOCAL_COPY = True  #@param {type: "boolean"}

if USE_LOCAL_COPY:
    LOCAL_DATA = '/content/data/arkitscenes'
    if not os.path.exists(f"{LOCAL_DATA}/images"):
        print("Copying data from Drive to local SSD...")
        !cp -r {DATA_DIR} {LOCAL_DATA}
        print("Done!")
    else:
        print("Local copy already exists.")
    TRAIN_DATA_ROOT = LOCAL_DATA
else:
    TRAIN_DATA_ROOT = DATA_DIR

print(f"Training data: {TRAIN_DATA_ROOT}")

## 5. Login to Weights & Biases

In [ ]:
!uv run wandb login

## 6. Train!

In [ ]:
#@title Training Config { run: "auto" }
RFDETR_VARIANT = "base"  #@param ["nano", "small", "base", "medium", "large"]
NUM_CLASSES = 17  #@param {type: "integer"}
PHASE = 1  #@param [1, 2] {type: "raw"}
EPOCHS = 24  #@param {type: "integer"}
BATCH_SIZE = 4  #@param {type: "integer"}
LR = 1e-4  #@param {type: "number"}
TARGET_SIZE = 560  #@param {type: "integer"}

# Save checkpoints to Drive (persists!)
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/rfdet3d-{RFDETR_VARIANT}-phase{PHASE}"

# Optional: WildDet3D checkpoint for 3D head weight transfer
WILDDET3D_CKPT = ""  # Set to path on Drive if you have it

print(f"Variant: {RFDETR_VARIANT}")
print(f"Phase: {PHASE} ({'RF-DETR frozen' if PHASE == 1 else 'all trainable'})")
print(f"Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, LR: {LR}")
print(f"Data: {TRAIN_DATA_ROOT}")
print(f"Checkpoints → {OUTPUT_DIR} (on Drive)")

In [ ]:
ckpt_flag = f"--wilddet3d_ckpt {WILDDET3D_CKPT}" if WILDDET3D_CKPT else ""

!uv run python train.py \
    --rfdetr_variant {RFDETR_VARIANT} \
    --num_classes {NUM_CLASSES} \
    --data_root {TRAIN_DATA_ROOT} \
    --target_size {TARGET_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --phase {PHASE} \
    --wandb_project rfdet3d \
    --wandb_name rfdet3d-{RFDETR_VARIANT}-phase{PHASE} \
    --output_dir {OUTPUT_DIR} \
    --num_workers 2 \
    --device cuda \
    {ckpt_flag}

## 7. Test inference with trained model

In [ ]:
import glob

# Find best checkpoint
best_ckpt = f"{OUTPUT_DIR}/rfdet3d_best.pt"
if os.path.exists(best_ckpt):
    print(f"Best checkpoint: {best_ckpt}")
    ckpt_size = os.path.getsize(best_ckpt) / 1e6
    print(f"Size: {ckpt_size:.1f} MB")
else:
    ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/rfdet3d_epoch*.pt"))
    if ckpts:
        best_ckpt = ckpts[-1]
        print(f"Using latest: {best_ckpt}")
    else:
        print("No checkpoints found. Run training first.")

In [ ]:
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Quick reload for development
import importlib
import wilddet3d_rfdetr.inference as inf_mod
importlib.reload(inf_mod)
from wilddet3d_rfdetr.inference import preprocess
from wilddet3d_rfdetr import RFDet3D, RFDet3DInput

# Load model
model = RFDet3D(
    rfdetr_variant=RFDETR_VARIANT,
    num_classes=NUM_CLASSES,
    score_threshold=0.3,
)
ckpt = torch.load(best_ckpt, map_location='cpu')
model.load_state_dict(ckpt['state_dict'])
model = model.cuda().eval()

# Pick a test image
test_images = glob.glob(f"{TRAIN_DATA_ROOT}/images/*.jpg")[:1]
if test_images:
    img = np.array(Image.open(test_images[0]).convert('RGB')).astype(np.float32)
    data = preprocess(img, target_size=TARGET_SIZE)

    with torch.no_grad():
        batch = RFDet3DInput(
            images=data['images'].cuda(),
            intrinsics=data['intrinsics'].cuda(),
            original_hw=[data['original_hw']],
            padding=[data['padding']],
        )
        out = model(batch)

    print(f"Detections: {len(out.boxes[0])}")

    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img.astype(np.uint8) if img.max() > 1 else img)
    for box, score, cls_id in zip(out.boxes[0].cpu(), out.scores[0].cpu(), out.class_ids[0].cpu()):
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor='lime', linewidth=2))
        ax.text(x1, y1-5, f'{score:.2f}', color='lime', fontsize=8)
    ax.set_title(f'{len(out.boxes[0])} detections')
    plt.show()
else:
    print('No test images found')